# Worker exit and timeout regression tests

Setup is explicit; existing assets are verified before reuse.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
import json,sys,tempfile,unittest,socket
from pathlib import Path
from experiment_supervision import execute_stage
class Checks(unittest.TestCase):
    def test_success_exit_and_timeout_are_distinct(self):
        # Bind only to discover an available test port; never terminate any external process.
        with socket.socket() as sock:
            sock.bind(('127.0.0.1',0));worker=sock.getsockname()[1]-5005
        with tempfile.TemporaryDirectory() as tmp:
            root=Path(tmp);stage=root/'success';stage.mkdir()
            command=[sys.executable,'-c',"import pathlib,json;pathlib.Path("+repr(str(stage/'result.json'))+").write_text(json.dumps({'status':'PASS'}))"]
            result=execute_stage(command,stage,10,worker)
            self.assertEqual(result['exit_code'],0);self.assertFalse(result['timed_out'])
            stage=root/'timeout';stage.mkdir()
            with self.assertRaises(RuntimeError):execute_stage([sys.executable,'-c','import time;time.sleep(60)'],stage,.05,worker)
            result=json.loads((root/'timeout-supervision.json').read_text())
            self.assertTrue(result['timed_out']);self.assertNotEqual(result['exit_code'],0)
suite=unittest.defaultTestLoader.loadTestsFromTestCase(Checks)
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
print('Worker supervision cases passed:',result.testsRun)


test_success_exit_and_timeout_are_distinct (__main__.Checks.test_success_exit_and_timeout_are_distinct) ... 

Frozen runtime contract definitions/execution completed.
Exclusive attempts and supervised worker processes definitions/execution completed.


ok


----------------------------------------------------------------------
Ran 1 test in 0.410s

OK


Worker supervision cases passed: 1
